<a href="https://colab.research.google.com/github/gaurinandwana/JPMORGAN_research/blob/main/loan_data_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from xgboost import XGBClassifier

# ==========================================
# 1. GENERATING CHARLIE'S PROTOTYPE DATA
# ==========================================
# Simulating a realistic 5,000 row retail loan sample based on your description
np.random.seed(42)
n_customers = 5000

mock_loan_book = {
    'customer_id': range(10001, 10001 + n_customers),
    'annual_income': np.random.normal(60000, 18000, n_customers).clip(15000),
    'loan_amount_outstanding': np.random.exponential(15000, n_customers) + 2000,
    'employment_length_years': np.random.randint(0, 20, n_customers),
    'debt_to_income_ratio': np.random.uniform(0.05, 0.65, n_customers),
    'historical_delinquencies': np.random.choice([0, 1, 2, 3], p=[0.85, 0.10, 0.04, 0.01], size=n_customers)
}
df = pd.DataFrame(mock_loan_book)

# Create a synthetic target 'defaulted_next_year' influenced heavily by DTI, delinquencies, and low income
log_odds = (
    (df['debt_to_income_ratio'] * 6.5) +
    (df['historical_delinquencies'] * 1.2) -
    (df['annual_income'] * 0.00003) -
    (df['employment_length_years'] * 0.08)
)
pd_true = 1 / (1 + np.exp(-log_odds))
df['defaulted_next_year'] = np.where(pd_true > np.random.uniform(0, 1, n_customers), 1, 0)

# ==========================================
# 2. MODEL PREPARATION & TRAINING
# ==========================================
features = ['annual_income', 'loan_amount_outstanding', 'employment_length_years', 'debt_to_income_ratio', 'historical_delinquencies']
X = df[features]
y = df['defaulted_next_year']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Scale features for the Logistic Regression pipeline
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model 1: Logistic Regression (Highly Interpretability, Great Baseline)
lr_model = LogisticRegression(class_weight='balanced', random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

# Model 2: XGBoost (Captures complex customer risk patterns)
xgb_model = XGBClassifier(scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]), eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

# Quick Evaluation Table for Charlie
print("==== PROTOTYPE MODEL PERFORMANCE ====")
print(f"Logistic Regression - AUC-ROC: {roc_auc_score(y_test, lr_probs):.4f} | Brier Score: {brier_score_loss(y_test, lr_probs):.4f}")
print(f"XGBoost Classifier  - AUC-ROC: {roc_auc_score(y_test, xgb_probs):.4f} | Brier Score: {brier_score_loss(y_test, xgb_probs):.4f}\n")

# ==========================================
# 3. EXPECTED LOSS PRODUCTION FUNCTION
# ==========================================
class RetailLoanLossEvaluator:
    def __init__(self, model, scaler=None, recovery_rate=0.10):
        self.model = model
        self.scaler = scaler
        self.lgd = 1.0 - recovery_rate  # Loss Given Default

    def evaluate_expected_loss(self, customer_profile: dict) -> dict:
        """
        Calculates PD, EAD, and Expected Loss for a single retail customer profile.
        """
        input_df = pd.DataFrame([customer_profile])
        ordered_features = ['annual_income', 'loan_amount_outstanding', 'employment_length_years', 'debt_to_income_ratio', 'historical_delinquencies']
        input_df = input_df[ordered_features]

        # Exposure at Default is the outstanding loan amount
        ead = customer_profile['loan_amount_outstanding']

        # Predict Probability of Default (PD)
        if self.scaler:
            scaled_input = self.scaler.transform(input_df)
            pd_val = self.model.predict_proba(scaled_input)[0][1]
        else:
            pd_val = self.model.predict_proba(input_df)[0][1]

        # Capital Math: EL = EAD * PD * LGD
        expected_loss = ead * pd_val * self.lgd

        return {
            'Estimated PD': f"{pd_val * 100:.2f}%",
            'Exposure (EAD)': f"${ead:,.2f}",
            'Loss Given Default (LGD)': f"{self.lgd * 100:.0f}%",
            'Expected Loss Provision': f"${expected_loss:,.2f}"
        }

# ==========================================
# 4. TESTING THE PROTOTYPE
# ==========================================
# Deploying the XGBoost model into our loss evaluator
loss_calculator = RetailLoanLossEvaluator(model=xgb_model, scaler=None, recovery_rate=0.10)

# Scenario: A high-risk retail borrower profile
charlie_test_profile = {
    'annual_income': 42000,
    'loan_amount_outstanding': 18500,
    'employment_length_years': 2,
    'debt_to_income_ratio': 0.58,
    'historical_delinquencies': 2
}

results = loss_calculator.evaluate_expected_loss(charlie_test_profile)

print("==== SAMPLE CUSTOMER RISK EVALUATION ====")
for metric, val in results.items():
    print(f"{metric}: {val}")

==== PROTOTYPE MODEL PERFORMANCE ====
Logistic Regression - AUC-ROC: 0.8194 | Brier Score: 0.1737
XGBoost Classifier  - AUC-ROC: 0.7818 | Brier Score: 0.1968

==== SAMPLE CUSTOMER RISK EVALUATION ====
Estimated PD: 99.76%
Exposure (EAD): $18,500.00
Loss Given Default (LGD): 90%
Expected Loss Provision: $16,609.55
